# Building an Object-Detection Training Set from Videos

This demo turns raw videos into a PyTorch-ready object-detection training set with a declarative Pixeltable schema.

```
videos (table)       source videos + metadata
  -> frames (view)   one frame per second + YOLOX pseudo-labels
  -> training_frames frames with detections, resized to 640x640
```

Install the project once with `uv sync`, then run this notebook with `uv run jupyter lab`.

## Create the schema

The pipeline lives in `schema.py`. `pxt schema update` creates missing tables and updates existing ones.

In [ ]:
from pathlib import Path

print(Path("schema.py").read_text())

In [ ]:
!uv run pxt schema update schema.py od_demo

In [ ]:
import pixeltable as pxt

videos = pxt.get_table("od_demo.videos")
frames = pxt.get_table("od_demo.frames")
training_frames = pxt.get_table("od_demo.training_frames")

## Insert videos

These public videos come from the [Multimedia Commons](http://mmcommons.org/) S3 bucket. Inserting them runs frame extraction, YOLOX inference, and training-sample construction.

In [ ]:
S3_PREFIX = "s3://multimedia-commons/data/videos/mp4/"
video_paths = [
    "9cd/047/9cd047335f12962a5e633c3b9e6602e.mp4",
    "111/015/1110153dc81d833025828f875b3ad2.mp4",
    "888/061/8880616a157cfd9036fc16ad6cf5c3af.mp4",
    "abc/01a/abc01aa533364ca1597e0f287f393b9.mp4",
    "00a/a5b/00aa5b8cc17f67eded6a257a61119.mp4",
    "7ab/023/7ab02326a7c9ab4728681f63aaa7c36d.mp4",
]

videos.insert([{"video": S3_PREFIX + path} for path in video_paths])
print(f"{videos.count()} videos -> {frames.count()} frames -> {training_frames.count()} training samples")

## Inspect the pseudo-labels

`overlay` renders the YOLOX detections on each frame and stores the result in Backblaze B2.

In [ ]:
frames.select(frames.overlay, frames.num_detections, frames.detections).head(2)

## Incremental computation

Insert one more video. Only its new frames are processed; existing results remain untouched.

In [ ]:
videos.insert([{"video": S3_PREFIX + "cde/078/cde078a6803a16cae79eb438e178f0ce.mp4"}])
print(f"{videos.count()} videos -> {frames.count()} frames -> {training_frames.count()} training samples")

## Image transforms for detection training

Pixeltable caches deterministic preprocessing such as resizing images and boxes. Stochastic augmentation belongs in the DataLoader so it changes every epoch.

In [ ]:
training_frames.select(training_frames.training_overlay, training_frames.labels).head(2)

## Export to PyTorch

`to_pytorch_dataset()` caches the query result and returns an `IterableDataset`. The custom collator handles variable-length boxes and applies stochastic augmentation.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2

dataset = training_frames.select(
    training_frames.image, training_frames.boxes, training_frames.labels
).to_pytorch_dataset(image_format="pt")

augment = v2.RandomHorizontalFlip(p=0.5)

def collate(samples: list[dict]) -> dict:
    images, boxes, labels = [], [], []
    for sample in samples:
        bounding_boxes = tv_tensors.BoundingBoxes(
            torch.tensor(sample["boxes"], dtype=torch.float32),
            format="XYXY",
            canvas_size=(640, 640),
        )
        image, bounding_boxes = augment(sample["image"], bounding_boxes)
        images.append(image)
        boxes.append(torch.as_tensor(bounding_boxes))
        labels.append(torch.tensor(sample["labels"], dtype=torch.int64))
    return {"images": torch.stack(images), "boxes": boxes, "labels": labels}

loader = DataLoader(dataset, batch_size=8, collate_fn=collate)
batch = next(iter(loader))
print("images:", batch["images"].shape, batch["images"].dtype)
print("boxes[0]:", batch["boxes"][0].shape)
print("labels[0]:", batch["labels"][0].shape)

## Where to go from here

- Export the same data as COCO JSON with `to_coco_dataset()`.
- Add an embedding index to curate more frames like a hard example.
- Label with a larger model and evaluate a smaller model against it.